# Input values
hi

In [ ]:
import excel_reader
from typing import Dict, List, Tuple

# General
Version = 'CH'
savefiles = True
N_points = 1000                                                 # Number of data points for the curves
run_environment = True
write_to_file = False
out_filename = "output_CH.csv"

# -----------------------------------------------------------

# Propellant tanks geometry
#d_int = 0.200058109                                             # [m], tank diameter
#h_cyl = 0.509                                                   # [m], height of the tank cylindrical part
r_int = 0.115                                                   # [m], tank diameter
h_cyl = 0.3851008075                                            # [m], height of the tank cylindrical part

h_cap = 0.05                                                    # [m], height of the tank bulkhead ellipse

# -----------------------------------------------------------

# Hold-down
## Geometry
alpha = 6                                                                                   # [°], angle between launch vehicle (launch rail) and gravity vector
beta = 6                                                                                    # [°], angle between launch vehicle axis and hold down cable axis
l_rail = 11.65                                                                              # [m], launch rail length

## other
g = 9.81                                                                                    # [m/s^2], gravity
F_HD_break = 3300                                                                           # [N], force at which the hold-down pin breaks (0 : no hold-down)
mu = 0.5                                                                                    # [-], friction coefficient between the rail and rail-button
m_dry = 108                                                                                 # [kg], launch vehicle dry mass (weighted 84.4 w. 3U)
m_additions = 0                                                                             # [kg] additions of new mass components since rocket weighting

# -----------------------------------------------------------

# Propellant Masses
prop_masses:Dict[str,float] = {}

OF_ratio = 1.5                                                                              # [-]
ISP = 220                                                                                   # [s]
Thrust = 6500                                                                               # [N]
Total_impulse = 40500                                                                       # [Ns] Modify this value (from: 36250 config 6 in https://docs.google.com/spreadsheets/d/1_r804lrg8Qi8M8p9dzCOJfm_6ZMJX5gqe4yg1-POdDs/edit?gid=1764341437#gid=1764341437)

fraction_film_cooling = 0.0750                                                              # [-]

mass_flow_rate = Thrust / (ISP * g)                                                         # [kg/s]

mass_flow_rate_ethanol = mass_flow_rate / (1 + OF_ratio)                                    # [kg/s]
mass_flow_rate_ethanol_ignition = 0                                                         # [kg/s]
mass_flow_rate_film_cooling = mass_flow_rate_ethanol * fraction_film_cooling                # [kg/s]
mass_flow_rate_ethanol_burn = mass_flow_rate_ethanol + mass_flow_rate_film_cooling          # [kg/s]

mass_flow_rate_lox_boil_off = 0.001                                                         # [kg/s]
mass_flow_rate_lox_ignition = 4.362                                                         # [kg/s]
mass_flow_rate_lox_prechill = 4.362                                                         # [kg/s]
mass_flow_rate_lox_burn = mass_flow_rate / (1 + 1/OF_ratio)                                 # [kg/s]

burn_time = Total_impulse / Thrust                                                          # [s]
propellant_mass = mass_flow_rate * burn_time                                                # [kg]

hold_time = 300                                                                             # [s]
prechill_time = 0.200                                                                       # [s]
ignition_delay = -0.050                                                                     # [s]
ramp_up_time = 0.400                                                                        # [s]
cutoff_time = 0.025                                                                         # [s]
ramp_down_time = 0.200                                                                      # [s]

## Fuel (ethanol)
m_fuel_delay = 0                                                                            # [kg]
m_fuel_cutoff = mass_flow_rate_ethanol_burn * cutoff_time                                   # [kg]
m_fuel_burn = mass_flow_rate_ethanol_burn * burn_time                                       # [kg], fuel mass corresponding to free volume
m_fuel_end = 0.405 + m_fuel_cutoff                                                          # [kg], fixed volume (we add m_fuel_cutoff since it is in the tank during the flight and technically used at the end of the burn without producing thust, we accept the extra mass for recovery phase)
m_fuel_total = m_fuel_delay + m_fuel_burn + m_fuel_end                                      # [kg], total mass in tanks
rho_fuel = 810                                                                              # [kg/m^3], fuel density

## Oxidizer (lox)
m_ox_boil_off = mass_flow_rate_lox_boil_off * hold_time                                     # [kg]
m_ox_prechill = mass_flow_rate_lox_prechill * prechill_time                                 # [kg]
m_ox_delay = mass_flow_rate_lox_ignition * abs(ignition_delay)                              # [kg]
m_ox_end = 0.577                                                                            # [kg], fixed volume
m_ox_burn = mass_flow_rate_lox_burn * burn_time                                             # [kg], oxidizer mass corresponding to free volume
m_ox_total = m_ox_boil_off + m_ox_prechill + m_ox_delay + m_ox_burn + m_ox_end              # [kg], total mass in tanks
rho_ox = 1154                                                                               # [kg/m^3], oxidizer density

## Pressurant (N2)
m_n2_copv = 5.210/2                                                                         # [kg], pressurant mass fixed volume


## Wet mass (on pad right after filling)
wet_mass = m_dry + m_fuel_total + m_ox_total + 2*m_n2_copv                                  # [kg]

# -----------------------------------------------------------

# thrust Curve
## Force
F_full_thrust = Thrust                                                                      # [N], peak thrust
F_derating = Thrust                                                                         # [N], constant thrust after ramp-up until derating starts
F_ramp_down = Thrust                                                                        # [N], thrust at the end of the derating phase
F_shutdown = 0                                                                              # [N], no thrust delivered at the end

## Timings
t_full_thrust = ramp_up_time                                                                # [s], ramp-up duration
t_ramp_down = burn_time                                                                     # [s], time at which ramp-down starts
t_derating = t_ramp_down-0.001                                                              # [s], time at which constant thrust stops and derating starts
t_shutdown = t_ramp_down+ramp_down_time  